# Google Maps No-Website Business Leads

Notebook ini mencari bisnis di Google Maps melalui **Playwright browser automation**, membersihkan data, menghapus duplikat, lalu **hanya menyimpan bisnis yang belum memiliki website**.

Kategori default:
- Salon kuku
- Dokter
- Klinik perawatan kulit
- Restoran
- Klinik gigi
- Hotel
- Hostel
- Pusat kebugaran

Output:
- CSV
- Excel/XLSX
- Opsional: kirim langsung ke Google Sheets

> **Catatan:** Google Maps dapat mengubah struktur UI kapan saja dan dapat menampilkan CAPTCHA/unusual-traffic page. Notebook ini tidak membypass CAPTCHA atau proteksi anti-bot. Gunakan jumlah hasil yang konservatif dan hanya gunakan data bisnis publik untuk analisis/prospecting yang wajar.


## 1. Install project + Playwright


In [ ]:
# Clone/update repository
import os, subprocess, pathlib

REPO_DIR = pathlib.Path("/content/prospect-client-analysis")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/softmitend/prospect-client-analysis.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)

# Python dependencies
!python -m pip install -q -r requirements.txt
!python -m pip install -q gspread gspread-dataframe

# Browser + Linux dependencies for Colab
!python -m playwright install --with-deps chromium

print("Ready:", os.getcwd())


## 2. Konfigurasi pencarian


In [ ]:
# ===== EDIT BAGIAN INI =====
LOCATION = "Banyuwangi, East Java, Indonesia"

# Batas per SEARCH QUERY, bukan per kategori.
# Mulai dari 10-20 untuk testing. Naikkan perlahan jika sudah stabil.
MAX_RESULTS_PER_QUERY = 20

# Colab menggunakan Chromium headless.
HEADLESS = True

# Canonical category -> variasi query untuk meningkatkan recall.
CATEGORY_QUERIES = {
    "Salon kuku": [
        "salon kuku",
        "nail salon",
    ],
    "Dokter": [
        "dokter",
        "doctor clinic",
    ],
    "Klinik perawatan kulit": [
        "klinik perawatan kulit",
        "skin care clinic",
        "dermatology clinic",
    ],
    "Restoran": [
        "restoran",
        "restaurant",
    ],
    "Klinik gigi": [
        "klinik gigi",
        "dental clinic",
        "dentist",
    ],
    "Hotel": [
        "hotel",
    ],
    "Hostel": [
        "hostel",
    ],
    "Pusat kebugaran": [
        "pusat kebugaran",
        "gym",
        "fitness center",
    ],
}

print("Location:", LOCATION)
print("Canonical categories:", len(CATEGORY_QUERIES))
print("Search queries:", sum(len(v) for v in CATEGORY_QUERIES.values()))


## 3. Scrape Google Maps

Setiap variasi query dijalankan terpisah. Setelah itu semua hasil digabung dan kategori diseragamkan ke kategori canonical di atas.

Karena beberapa query dapat menemukan bisnis yang sama, deduplikasi dilakukan pada tahap preprocessing.


In [ ]:
from src.google_maps_browser import GoogleMapsPlaywrightScraper

scraper = GoogleMapsPlaywrightScraper(
    headless=HEADLESS,
    browser_channel=None,  # Playwright Chromium
    detail_pause_seconds=0.7,
    scroll_pause_seconds=1.0,
    max_idle_scrolls=4,
)

raw_rows = []

for canonical_category, aliases in CATEGORY_QUERIES.items():
    for alias in aliases:
        print(f"\n🔎 {canonical_category} | query: {alias!r}")

        rows = scraper.collect_categories(
            location=LOCATION,
            categories=[alias],
            max_results_per_category=MAX_RESULTS_PER_QUERY,
            language_code="id",
            region_code="ID",
        )

        # Simpan kategori target yang konsisten untuk analisis/export.
        for item in rows:
            item["_requested_category"] = canonical_category
            item["_source_query"] = f"{alias} in {LOCATION}"

        raw_rows.extend(rows)
        print("   found:", len(rows))

print("\nTotal raw rows:", len(raw_rows))


## 4. Preprocessing + ETL + filter `NO WEBSITE`


In [ ]:
import pandas as pd
import numpy as np

from src.etl import places_to_frame, preprocess_businesses

df = places_to_frame(raw_rows)

if df.empty:
    raise RuntimeError(
        "Tidak ada data yang terkumpul. Coba kurangi kategori, cek LOCATION, "
        "atau jalankan ulang jika Google menampilkan halaman berbeda."
    )

# Simpan kategori asli Google Maps sebelum kita seragamkan.
df["maps_category"] = df["category"]
df["category"] = df["requested_category"].fillna(df["category"])

# Cleaning + deduplication dari pipeline project.
clean = preprocess_businesses(df)

# Normalisasi website kosong.
clean["website"] = (
    clean["website"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "<NA>": pd.NA, "None": pd.NA})
)

clean["website_present"] = clean["website"].notna()

# TARGET UTAMA: hanya bisnis tanpa website.
leads = clean.loc[~clean["website_present"]].copy()

# Google Maps biasanya tidak menyediakan email/WhatsApp sebagai field langsung.
# Kolom tetap dibuat agar spreadsheet konsisten dan bisa dilengkapi manual.
leads["email"] = pd.NA
leads["whatsapp"] = pd.NA

# Feature sederhana untuk membantu prioritas outreach.
leads["has_phone"] = leads["phone"].notna()
leads["has_address"] = leads["address"].notna()
leads["has_hours"] = leads["opening_hours_text"].notna()

leads["contact_score"] = (
    leads["has_phone"].astype(int) * 50
    + leads["has_address"].astype(int) * 30
    + leads["has_hours"].astype(int) * 20
)

# Prioritaskan bisnis tanpa website tetapi masih mudah dihubungi.
sort_cols = ["contact_score", "review_count", "rating"]
for col in sort_cols:
    if col not in leads:
        leads[col] = np.nan

leads = leads.sort_values(
    sort_cols,
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)

# Kolom output yang rapi untuk spreadsheet.
OUTPUT_COLUMNS = [
    "business_name",
    "category",
    "maps_category",
    "phone",
    "email",
    "whatsapp",
    "address",
    "opening_hours_text",
    "rating",
    "review_count",
    "business_status",
    "google_maps_url",
    "latitude",
    "longitude",
    "contact_score",
    "source_query",
]

OUTPUT_COLUMNS = [c for c in OUTPUT_COLUMNS if c in leads.columns]
leads_export = leads[OUTPUT_COLUMNS].copy()

print(f"Unique businesses after preprocessing: {len(clean):,}")
print(f"Businesses WITHOUT website: {len(leads_export):,}")
display(leads_export.head(30))


## 5. Ringkasan per kategori


In [ ]:
summary = (
    leads_export
    .groupby("category", dropna=False)
    .agg(
        businesses_without_website=("business_name", "count"),
        with_phone=("phone", lambda s: int(s.notna().sum())),
        avg_rating=("rating", "mean"),
        median_reviews=("review_count", "median"),
        avg_contact_score=("contact_score", "mean"),
    )
    .reset_index()
)

summary["phone_coverage_pct"] = (
    summary["with_phone"] / summary["businesses_without_website"] * 100
).round(1)

summary["avg_rating"] = summary["avg_rating"].round(2)
summary["avg_contact_score"] = summary["avg_contact_score"].round(1)

summary = summary.sort_values(
    ["businesses_without_website", "phone_coverage_pct"],
    ascending=[False, False],
    ignore_index=True,
)

display(summary)


## 6. Export CSV + Excel


In [ ]:
from pathlib import Path

OUT_DIR = Path("/content")
CSV_PATH = OUT_DIR / "businesses_without_website.csv"
XLSX_PATH = OUT_DIR / "businesses_without_website.xlsx"

leads_export.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as writer:
    leads_export.to_excel(writer, sheet_name="leads_no_website", index=False)
    summary.to_excel(writer, sheet_name="category_summary", index=False)

print("CSV :", CSV_PATH)
print("XLSX:", XLSX_PATH)


### Download ke komputer


In [ ]:
from google.colab import files

# Jalankan salah satu atau keduanya.
files.download(str(CSV_PATH))
# files.download(str(XLSX_PATH))


## 7. Opsional: kirim langsung ke Google Sheets

Cell ini meminta autentikasi akun Google Colab kamu.

- Isi `SPREADSHEET_URL` kalau ingin memakai spreadsheet yang sudah ada.
- Biarkan kosong untuk membuat spreadsheet baru di Google Drive.


In [ ]:
SPREADSHEET_URL = ""  # contoh: https://docs.google.com/spreadsheets/d/xxxxx/edit
NEW_SPREADSHEET_NAME = "Business Leads - No Website"

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

creds, _ = default()
gc = gspread.authorize(creds)

if SPREADSHEET_URL.strip():
    sh = gc.open_by_url(SPREADSHEET_URL.strip())
else:
    sh = gc.create(NEW_SPREADSHEET_NAME)

# Leads worksheet
try:
    ws = sh.worksheet("leads_no_website")
    ws.clear()
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(
        title="leads_no_website",
        rows=max(1000, len(leads_export) + 20),
        cols=max(20, len(leads_export.columns) + 5),
    )

set_with_dataframe(ws, leads_export, include_index=False, resize=True)

# Summary worksheet
try:
    ws_summary = sh.worksheet("category_summary")
    ws_summary.clear()
except gspread.WorksheetNotFound:
    ws_summary = sh.add_worksheet(
        title="category_summary",
        rows=max(100, len(summary) + 20),
        cols=max(15, len(summary.columns) + 5),
    )

set_with_dataframe(ws_summary, summary, include_index=False, resize=True)

print("Google Sheets ready:")
print(sh.url)


## Interpretasi hasil

`contact_score` bukan model prediktif. Ini hanya skor prioritas sederhana:

- +50 bila ada nomor telepon
- +30 bila ada alamat
- +20 bila ada jam operasional

Semua bisnis di output sudah memenuhi filter **tidak memiliki website pada profil Google Maps yang berhasil dibaca saat scraping**.

Ini bukan bukti absolut bahwa bisnis tersebut tidak punya website di internet. Bisa saja websitenya ada tetapi belum dicantumkan di profil Maps, atau selector Google Maps berubah sehingga field website tidak terbaca. Untuk prospecting, perlakukan kolom ini sebagai **lead signal**, bukan kebenaran mutlak.
